In [1]:
import os
from pathlib import Path
from typing import Optional

import fastmri
import h5py
import matplotlib.pyplot as plt
import numpy as np
import torch
from data_utils import *
from datasets import *
from fastmri.data.transforms import tensor_to_complex_np
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from torch.utils.data import DataLoader, TensorDataset

from model import *
from torch.optim import SGD, Adam, AdamW
from train_utils import *

import plotly.graph_objects as go

In [18]:
path_to_data = '/itet-stor/mcrespo/bmicdatasets-originals/Originals/fastMRI/brain/multicoil_val/'
n_volumes = 4
n_slices = 2
with_mask = False  # NOTE: During inference phase, set to True.
acceleration = 4
center_frac= 0.15
vol_id0= 0
# dataset = KCoordDataset(path_to_data=path_to_data, n_volumes=4, n_slices=2, acceleration=4)



In [20]:
path_to_data = Path(path_to_data)
if path_to_data.is_dir():
    files = sorted(
        [
            file
            for file in path_to_data.iterdir()
            if file.suffix == ".h5" and "AXT1POST_205" in file.name
            # if file.suffix == ".h5" and "AXT2_205" in file.name # T2 sequence
            
        ]
    )[:n_volumes]
else:
    files = [path_to_data]

# For each MRI volume in the dataset...
for vol_id, file in enumerate(files):
    # Load MRI volume
    with h5py.File(file, "r") as hf:
        volume_kspace = to_tensor(preprocess_kspace(hf["kspace"][()]))[
            :n_slices
        ]
    print(file)

/itet-stor/mcrespo/bmicdatasets-originals/Originals/fastMRI/brain/multicoil_val/file_brain_AXT1POST_205_2050004.h5
/itet-stor/mcrespo/bmicdatasets-originals/Originals/fastMRI/brain/multicoil_val/file_brain_AXT1POST_205_2050016.h5
/itet-stor/mcrespo/bmicdatasets-originals/Originals/fastMRI/brain/multicoil_val/file_brain_AXT1POST_205_2050017.h5
/itet-stor/mcrespo/bmicdatasets-originals/Originals/fastMRI/brain/multicoil_val/file_brain_AXT1POST_205_2050032.h5


In [17]:
ground_truth = []
kspace_gt = []
for id in range(len(dataset.metadata)):
    file = dataset.metadata[id]['file']
    print(file)
    # with h5py.File(file, "r") as hf:
    #     ground_truth.append(hf["reconstruction_rss"][()][
    #         : 2])
    #     kspace_gt.append(to_tensor(preprocess_kspace(hf["kspace"][()][: 2])))



/itet-stor/mcrespo/bmicdatasets-originals/Originals/fastMRI/brain/multicoil_train/file_brain_AXT1POST_205_2050005.h5
/itet-stor/mcrespo/bmicdatasets-originals/Originals/fastMRI/brain/multicoil_train/file_brain_AXT1POST_205_2050009.h5
/itet-stor/mcrespo/bmicdatasets-originals/Originals/fastMRI/brain/multicoil_train/file_brain_AXT1POST_205_2050014.h5
/itet-stor/mcrespo/bmicdatasets-originals/Originals/fastMRI/brain/multicoil_train/file_brain_AXT1POST_205_2050021.h5


In [16]:
### Compute the baselines
vol_id = 0
volume_kspace = torch.zeros_like(kspace_gt[vol_id])
##################################################
# Mask creation
##################################################
mask_func = EquiSpacedMaskFunc(
    center_fractions=[center_frac], accelerations=[acceleration]
)
shape = (1,) * len(volume_kspace.shape[:-3]) + tuple(
    volume_kspace.shape[-3:]
)
mask, _ = mask_func(
    shape, None, vol_id
)  # use the volume index as random seed.

mask, left_idx, right_idx = remove_center(mask)
#   # NOTE: Uncomment to include the center region in the training data. Note that 'left_idx' and 'right_idx' are still needed.


In [2]:
# checkpoint_metalearning = '/scratch_net/ken/mcrespo/proj_marina/logs/multivol_12_12/2024-12-12_15h18m48s/checkpoints/epoch_0499.pt'
checkpoint_15vols = '/scratch_net/ken/mcrespo/proj_marina/logs/multivol_12_10/2024-12-14_10h44m19s/checkpoints/epoch_0199.pt'
vol_embedding = torch.load(checkpoint_15vols,  map_location=torch.device('cpu'))["phi_vol"]["weight"]

/tmp/ipykernel_18211/900463937.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  vol_embedding = torch.load(checkpoint_15vols,  map_location=torch.device('cpu'))["phi_vol"

In [9]:
for i, param in enumerate(embeddings_coil.parameters()):
    print(i)
    print(param)

0
Parameter containing:
tensor([[ 0.5588,  0.9306, -0.2962,  ...,  0.2717, -0.4587,  0.4046],
        [-0.0087,  0.5309,  0.0846,  ...,  0.9300, -1.4168, -0.9138],
        [ 0.7271,  0.0578, -0.8004,  ..., -0.9159, -0.1959,  0.4468],
        ...,
        [-2.6225,  0.9015, -1.3069,  ...,  1.2771,  0.4491, -0.3794],
        [ 0.9795,  0.8902, -0.3777,  ..., -0.5107, -0.9940,  0.4320],
        [-1.0848, -0.5663,  0.8535,  ...,  0.9432,  1.6714,  0.0345]],
       requires_grad=True)


In [ ]:
print(embeddings_vol.weight.data[0][0], embeddings_vol.weight.data[1][0])

In [24]:
from pathlib import Path
path_to_data = Path('/itet-stor/mcrespo/bmicdatasets-originals/Originals/fastMRI/brain/multicoil_train/')
n_volumes = 15
vol_id0 = 0
if path_to_data.is_dir():
        files = sorted(
            [
                file
                for file in path_to_data.iterdir()
                if file.suffix == ".h5" and "AXT1POST_205" in file.name
            ]
        )[vol_id0:vol_id0+n_volumes]
        
ground_truth = []
kspace_gt = []
for i,vol in enumerate(files):
    file = vol
    with h5py.File(file, "r") as hf:
        ground_truth.append(
            hf["reconstruction_rss"][()][: 2]
        )
        
        kspace_gt.append(to_tensor(preprocess_kspace(hf["kspace"][()][: 2])))

In [8]:
vol = 0
path_to_data = Path('/itet-stor/mcrespo/bmicdatasets-originals/Originals/fastMRI/brain/multicoil_train/')
dataset = KCoordDataset(path_to_data, n_volumes=2, n_slices=2, with_mask=True, acceleration=4, center_frac=0.15)
center = dataset.metadata[vol]["center"]
left_idx, right_idx, center_vals = center["left_idx"], center["right_idx"], center["vals"]
shape = dataset.metadata[vol]["shape"]
n_slices, n_coils, height, width = shape

# Create tensors of indices for each dimension
kx_ids = torch.cat([torch.arange(left_idx), torch.arange(right_idx, width)])
ky_ids = torch.arange(height)
kz_ids = torch.arange(n_slices)
coil_ids = torch.arange(n_coils)

# Use meshgrid to create expanded grids
kspace_ids = torch.meshgrid(kx_ids, ky_ids, kz_ids, coil_ids, indexing="ij")
kspace_ids = torch.stack(kspace_ids, dim=-1).reshape(-1, len(kspace_ids))

dataset_pred = TensorDataset(kspace_ids)
dataloader_pred = DataLoader(
    dataset_pred, batch_size=60_000, shuffle=False, num_workers=3
)

# volume_kspace = torch.zeros(
#     (n_slices, n_coils, height, width, 2),
#     dtype=torch.float64,
# )  

# for point_ids in dataloader_pred:
#     # point_ids = point_ids[0].to(self.device, dtype=torch.long)
#     point_ids = torch.tensor(point_ids[0], dtype = torch.long) 

#     this_volume = kspace_gt[vol]
#     output = this_volume[point_ids[:,2], point_ids[:,3], point_ids[:,1], point_ids[:,0]]
#     volume_kspace[
#         point_ids[:, 2], point_ids[:, 3], point_ids[:, 1], point_ids[:, 0],
#     ] =  output

In [ ]:
# volume_kspace = tensor_to_complex_np(volume_kspace)
my_gt = tensor_to_complex_np(kspace_gt[vol])
mask = dataset.metadata[vol]["mask"].squeeze(-1).expand(shape).numpy()


my_data = gt*(mask)
my_predicted = tensor_to_complex_np(volume_kspace)*(1-mask)
my_final = my_data + my_predicted

y_data_rss = rss(my_data)
y_predicted_rss = rss(my_predicted)
y_final_rss = rss(my_final)

plt.figure(figsize=(10,20))
plt.subplot(1,3,1)
plt.imshow(np.abs(y_data_rss[0]))
plt.colorbar()
plt.subplot(1,3,2)
plt.imshow(np.abs(y_predicted_rss[0]))
plt.colorbar()
plt.subplot(1,3,3)
plt.imshow(np.abs(y_final_rss[0]))
plt.colorbar()

In [ ]:
kspace_gt[0].shape

In [ ]:
vol = 0
volume_kspace = np.load('/scratch_net/ken/mcrespo/proj_marina/logs/multivol_12_11_test/vol_kspace_unw.npy')
volume_kspace_unw = np.load('/scratch_net/ken/mcrespo/proj_marina/logs/multivol_12_11_test/vol_kspace_unw.npy')
gt = tensor_to_complex_np(kspace_gt[vol])

mask = np.load('/scratch_net/ken/mcrespo/proj_marina/logs/multivol_12_11_test/mask.npy')
cste_mod = np.load('/scratch_net/ken/mcrespo/proj_marina/logs/multivol_12_11_test/cste.npy')

####### mask
y_kspace_data_u = gt  * (mask)
y_kspace_prediction_u = volume_kspace_unw * (1-mask)
y_kspace_final = y_kspace_data_u + y_kspace_prediction_u

y_kspace_data_rss = rss(y_kspace_data_u)
y_kspace_prediction_rss = rss(y_kspace_prediction_u)
y_kspace_final_rss = rss(y_kspace_final)

y_kspace_final[..., left_idx:right_idx] = center_vals
y_img_final = np.abs(rss(inverse_fft2_shift(y_kspace_final)))

###### predict the edges - image
y_img_edges = np.abs(rss(inverse_fft2_shift(volume_kspace)))


###### predict the center - image
volume_kspace[..., left_idx:right_idx] = center_vals
y_img_edges_center = np.abs(rss(inverse_fft2_shift(volume_kspace)))

cste_arg = np.pi/180
# ###### raw img w/o 
# y_kspace_data[..., left_idx:right_idx] = 0
# raw_img_edges = np.abs(rss(inverse_fft2_shift(y_kspace_data)))

# cste_mod = dataset.metadata[vol]["norm_cste"]

fig = plt.figure(figsize=(10,10))
# for slice_id in range(shape[0]):


plt.subplot(1,3,1)
plt.imshow(np.abs(y_kspace_data_rss[0])/cste_mod)
plt.axis('off')
plt.title('M·ydata')
plt.subplot(1,3,2)
plt.imshow(np.abs(y_kspace_prediction_rss[0])/cste_mod)
plt.axis('off')
plt.title('(1-M)·ypred')
plt.subplot(1,3,3)                
plt.imshow(np.abs(y_kspace_final_rss[0])/cste_mod)
plt.axis('off')
plt.title('M·ydata + (1-M)·ypred')
    
# self.writer.add_figure(
#     f"prediction/vol_{vol_id}/slice_{slice_id}/kspace_composed",
#     fig,
#     global_step=epoch_idx,
# )
# plt.close(fig)


fig = plt.figure(figsize=(10,10))

plt.subplot(1,3,1)
plt.imshow(y_img_edges[0], cmap='gray')
plt.axis('off')
plt.title('Predicted edges')
plt.subplot(1,3,2)
plt.imshow(y_img_edges_center[0], cmap='gray')
plt.axis('off')
plt.title('Predicted edges + center')
plt.subplot(1,3,3)                
plt.imshow(y_img_final[0], cmap='gray')
plt.axis('off')
plt.title('Predicted + Acquired')
    
# self.writer.add_figure(
#     f"prediction/vol_{vol_id}/slice_{slice_id}/volume img",
#     fig,
#     global_step=epoch_idx,
# )
# plt.close(fig)
# volume_kspace_unweighted[..., left_idx:right_idx] = 0
volume_kspace[...,left_idx:right_idx] = center_vals
modulus = np.abs(rss(volume_kspace))
phase = np.angle(rss(volume_kspace))
fig = plt.figure(figsize=(10,10))
# for slice_id in range(shape[0]):
plt.subplot(1,2,1)
eps = 1.e-45
plt.imshow(modulus[0]/cste_mod)
plt.axis('off')
plt.colorbar()
plt.title('Acquired kspace')
plt.subplot(1,2,2)
plt.imshow(phase[0]/cste_arg)
plt.axis('off')
plt.title('Acquired img')
    

In [ ]:

y_data_rss = rss(y_data)
y_predicted_rss = rss(y_pred)
y_final_rss = rss(y_final)

plt.figure(figsize=(10,20))
plt.subplot(1,3,1)
plt.imshow(np.abs(y_data_rss[0]))
plt.colorbar()
plt.subplot(1,3,2)
plt.imshow(np.abs(y_predicted_rss[0]))
plt.colorbar()
plt.subplot(1,3,3)
plt.imshow(np.abs(y_final_rss[0]))
plt.colorbar()


In [ ]:
plt.figure(figsize=(10,20))
plt.subplot(1,3,1)
plt.imshow(np.abs(rss(my_data - y_data)[0]))
plt.axis('off')
plt.subplot(1,3,2)
plt.imshow(np.abs(rss(my_predicted - y_pred)[0]))
plt.axis('off')
plt.subplot(1,3,3)
plt.imshow(np.abs(rss(my_final - y_final)[0]))
plt.axis('off')

In [ ]:
for vol_id in range(15):
    gt_img = ground_truth[vol_id]
    plt.figure(figsize=(5,5))
    plt.imshow(gt_img[0], cmap='gray')
    plt.axis('off')
    plt.title(f"Vol: {vol_id}")

In [4]:
# model_checkpoint = '/scratch_net/ken/mcrespo/proj_marina/logs/multivol_12_04/2024-12-07_12h26m33s/checkpoints/epoch_0599.pt'  # TODO: SET (OR LEAVE COMMENTED).
model_checkpoint = '/scratch_net/ken/mcrespo/proj_marina/logs/multivol_12_10/2024-12-14_10h44m19s/checkpoints/epoch_0199.pt'

model = Siren(coord_dim=3, vol_embedding_dim=256, coil_embedding_dim=128, hidden_dim=512, n_layers=8, out_dim=2)
# Load checkpoint.
model_state_dict = torch.load(model_checkpoint,  map_location=torch.device('cpu'))["model_state_dict"]
embedding_coil = torch.load(model_checkpoint,  map_location=torch.device('cpu'))["embedding_coil_state_dict"]["weight"]
embedding_vol = torch.load(model_checkpoint,  map_location=torch.device('cpu'))["embedding_vol_state_dict"]["weight"]
optimizer = torch.load(model_checkpoint,  map_location=torch.device('cpu'))["optimizer_state_dict"]

/tmp/ipykernel_12505/1869319110.py:6: FutureWarning:

You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.

/tmp/ipykernel_12505/1869319110.py:7: FutureWarning:

You are using `torch.load` with `we

In [ ]:
# path_vol11 = '/scratch_net/ken/mcrespo/proj_marina/logs/multivol_12_10/2024-12-10_21h45m46s/checkpoints/epoch_0499.pt'
# path_vol5 ='/scratch_net/ken/mcrespo/proj_marina/logs/multivol_12_10/2024-12-10_21h48m45s/checkpoints/epoch_0499.pt'
# path_vol0 = '/scratch_net/ken/mcrespo/proj_marina/logs/multivol_12_04/2024-12-07_12h26m33s/checkpoints/epoch_0599.pt'

# paths123 = [path_vol0, path_vol5, path_vol11]


embeddings_vol = []
embeddings_coil = []

for path in paths123:
    embedd_dict = torch.load(path, map_location=torch.device('cpu'))["embedding_vol_state_dict"]["weight"]
    embedd_coil_dict = torch.load(path, map_location=torch.device('cpu'))["embedding_coil_state_dict"]["weight"]

    embeddings_vol.append(embedd_dict.data)
    embeddings_coil.append(embedd_coil_dict.data)
    
    
embeddings_vol = torch.cat(embeddings_vol, dim=0)
embeddings_coil = torch.cat(embeddings_coil, dim=0)



In [ ]:
plt.figure(figsize=(10,20))
for i in range(15):
    plt.subplot(5,3,(i+1))
    plt.imshow(ground_truth[i][0], cmap='gray')
    plt.title(f"Vol : {i+1}")
    plt.axis('off')

plt.tight_layout()

In [5]:
# Perform full SVD
embedding_vol -= embedding_vol.mean(0) # Demean the dictionary
U, S, Vt = np.linalg.svd(embedding_vol, full_matrices=False)
# Retain only the top k singular values and corresponding vectors
Vt_k = Vt[:, :3]
embeddings_vol_pca = np.matmul(U,(np.matmul(np.diag(S),Vt_k)))

U, S, Vt = np.linalg.svd(embedding_coil, full_matrices=False)
# Retain only the top k singular values and corresponding vectors
Vt_k = Vt[:, :3]
embeddings_coil_pca = np.matmul(U,(np.matmul(np.diag(S),Vt_k)))


# Generate 15 distinct colors using a colormap

colors = [
    "red", "blue", "green", "purple", "orange", "cyan", "magenta", "yellow",
    "black", "pink", "lime", "teal", "brown", "grey", "olive"
]



# Create the figure
fig = go.Figure()

###### Volume Embeddings
#######################################################
# Add each point as a separate trace with a unique color
for i in range(embeddings_vol_pca.shape[0]):
    fig.add_trace(go.Scatter3d(
        x=[embeddings_vol_pca[i, 0]],
        y=[embeddings_vol_pca[i, 1]],
        z=[embeddings_vol_pca[i, 2]],
        mode='markers',
        marker=dict(size=8, opacity=0.8, color=colors[i]),
        name=f"Point {i+1}"  # Legend entry for each point
    ))

# Update layout
fig.update_layout(
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z'
    ),
    title='PCA of 15 volume embeddings:',
    showlegend=True
)

# Show the plot
fig.show()

colors = [plt.cm.viridis(i / embeddings_coil_pca.shape[0]) for i in range(embeddings_coil_pca.shape[0])]  # Using the 'viridis' colormap
color_hex = [f'rgba({int(c[0]*255)},{int(c[1]*255)},{int(c[2]*255)},{c[3]})' for c in colors]  # Convert to Plotly-compatible RGBA
# Create the figure
fig = go.Figure()

###### Coil Embeddings
#######################################################
# Add each point as a separate trace with a unique color
for i in range(embeddings_coil_pca.shape[0]):
    fig.add_trace(go.Scatter3d(
        x=[embeddings_coil_pca[i, 0]],
        y=[embeddings_coil_pca[i, 1]],
        z=[embeddings_coil_pca[i, 2]],
        mode='markers',
        marker=dict(size=8, opacity=0.8, color=color_hex[i]),
        name=f"Point {i+1}"  # Legend entry for each point
    ))

# Update layout
fig.update_layout(
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z'
    ),
    title='PCA of 15 coil embeddings:',
    showlegend=True
)

# Show the plot
fig.show()



In [52]:
distances_vect = []
count = 0
saved_distances = []
saved_volumes = []

for i in range(embeddings_vol_pca.shape[0]):
    closest_vol_dist = 20
    closest_vol = 20
    for j in range(i, embeddings_vol_pca.shape[0]):
        
        if i != j:  
            print(f"-{count}-")
            print(f"Vol {i+1} vs Vol {j+1}: ")
            dist = torch.norm(torch.tensor(embeddings_vol_pca[i] - embeddings_vol_pca[j]))
            
            print(f"Distance : {dist}")
            distances_vect.append(dist)
            count += 1
            
            if closest_vol_dist > dist:
                closest_vol_dist = dist
                closest_vol = j+1
                
    saved_volumes.append(closest_vol)
    saved_distances.append(closest_vol_dist)
    

-0-
Vol 1 vs Vol 2: 
Distance : 0.018893688917160034
-1-
Vol 1 vs Vol 3: 
Distance : 0.019835960119962692
-2-
Vol 1 vs Vol 4: 
Distance : 0.010433133691549301
-3-
Vol 1 vs Vol 5: 
Distance : 0.012105301953852177
-4-
Vol 1 vs Vol 6: 
Distance : 0.006164844613522291
-5-
Vol 1 vs Vol 7: 
Distance : 0.012749423272907734
-6-
Vol 1 vs Vol 8: 
Distance : 0.02111239545047283
-7-
Vol 1 vs Vol 9: 
Distance : 0.008012855425477028
-8-
Vol 1 vs Vol 10: 
Distance : 0.006182783283293247
-9-
Vol 1 vs Vol 11: 
Distance : 0.014709495939314365
-10-
Vol 1 vs Vol 12: 
Distance : 0.0027220607735216618
-11-
Vol 1 vs Vol 13: 
Distance : 0.011698533780872822
-12-
Vol 1 vs Vol 14: 
Distance : 0.01156328059732914
-13-
Vol 1 vs Vol 15: 
Distance : 0.008412700146436691
-14-
Vol 2 vs Vol 3: 
Distance : 0.013503460213541985
-15-
Vol 2 vs Vol 4: 
Distance : 0.009723727591335773
-16-
Vol 2 vs Vol 5: 
Distance : 0.006978362333029509
-17-
Vol 2 vs Vol 6: 
Distance : 0.014953316189348698
-18-
Vol 2 vs Vol 7: 
Distance : 

In [53]:
saved_volumes

[12, 5, 11, 9, 7, 9, 11, 11, 10, 15, 12, 14, 14, 15, 20]

In [54]:
saved_distances

[tensor(0.0027),
 tensor(0.0070),
 tensor(0.0063),
 tensor(0.0042),
 tensor(0.0053),
 tensor(0.0033),
 tensor(0.0055),
 tensor(0.0122),
 tensor(0.0076),
 tensor(0.0041),
 tensor(0.0153),
 tensor(0.0092),
 tensor(0.0015),
 tensor(0.0190),
 20]

In [13]:
### 8 vs 3
dist = embeddings_vol_pca[8] - embeddings_vol_pca[3]
torch.norm(torch.tensor(dist))

tensor(0.0042)

In [15]:
### 5 vs 7

dist = embeddings_vol_pca[5] - embeddings_vol_pca[3]
torch.norm(torch.tensor(dist))

tensor(0.0071)

In [ ]:
### 6 vs 9

dist = embeddings_vol_pca[8] - embeddings_vol_pca[3]
torch.norm(torch.tensor(dist))

In [ ]:
### 6 vs 9

dist = embeddings_vol_pca[8] - embeddings_vol_pca[3]
torch.norm(torch.tensor(dist))

In [ ]:
path_to_data = '/itet-stor/mcrespo/bmicdatasets-originals/Originals/fastMRI/brain/multicoil_train/'
n_volumes = 5
n_slices = 2
with_mask= True  # NOTE: During inference phase, set to True.
with_center= False
acceleration= 4
vol_embedding_dim = 256
coil_embedding_dim = 128
center_frac= 0.15
embedding_dim = 256
sigma = 0.1

dataset = KCoordDataset(path_to_data, n_volumes=n_volumes, n_slices=n_slices, with_mask=with_mask, acceleration=acceleration, center_frac=center_frac)


In [ ]:
plt.figure(figsize=(20, 5))
for vol in range(5):
    mean = embedding_vol[vol].mean()
    plt.subplot(1, 5, vol+1)
    plt.hist(embedding_vol[vol], bins=100)
    plt.title(f"Vol: {vol}")
    # plt.axvline(mean, color = 'red', linewidth=.8)
    if vol != 0:
        plt.yticks([])
    
plt.show()

In [ ]:
## Comparison volume embeddings
similarities = []
for i, vol in enumerate(embedding_vol):
    for j, vol1 in enumerate(embedding_vol[:i]):
        
        cosine_similarity = np.dot(vol, vol1) / (np.linalg.norm(vol) * np.linalg.norm(vol1))
        euclidean_distance = np.linalg.norm(vol - vol1)
        
        
        print(f"{i} vs {j} : {euclidean_distance}") 
        # similarities.append(euclidean_distance)

In [ ]:
for  name, vec in enumerate(embedding_vol):
        print(f"Vol {name} ; Mean : {torch.mean(vec): .4f}, Variance {torch.var(vec): .4f}")

In [ ]:
count = 0
for i, param in enumerate(model.parameters()):
    if i == 8:
        print(f"Unfreezing parameters layer")
        param.requires_grad = True
        
    else:
        param.requires_grad = False
        
    count += 1

In [ ]:
my_vol(torch.tensor(0))[0]

In [ ]:
my_vol = torch.nn.Embedding(
        2, 256
    )
my_vol.weight.data.copy_(embedding_vol[:2])

In [ ]:
coil_optim = optimizer["state"][1]["exp_avg"]

In [ ]:

coil_optim

In [ ]:
path_to_data = '/itet-stor/mcrespo/bmicdatasets-originals/Originals/fastMRI/brain/multicoil_train/'
n_volumes= 1
n_slices= 2
with_mask= False  # NOTE: During inference phase, set to True.
with_center= False
acceleration= 4
vol_embedding_dim = 256
coil_embedding_dim = 128
center_frac= 0.15
embedding_dim = 256
sigma = 0.1

dataset = KCoordDataset(path_to_data, n_volumes=n_volumes, n_slices=n_slices, with_mask=with_mask, acceleration=acceleration, center_frac=center_frac)

dataloader = DataLoader(
    dataset,
    batch_size=120_000,
    num_workers=0,
    shuffle=True,
)

vol_id = 0
shape = dataloader.dataset.metadata[vol_id]["shape"]
center_data = dataloader.dataset.metadata[vol_id]["center"]
left_idx, right_idx, center_vals = (
    center_data["left_idx"],
    center_data["right_idx"],
    center_data["vals"],
)


# predicted_volume = volume_kspace.clone()

# for batch_idx, (inputs,inputs_unnormalized,targets) in enumerate(dataloader):
    
#     coords = inputs[:, 1:-1] # kx,ky,kz
#     vol_ids = inputs[:,0].long()
#     coil_ids = inputs[:,-1].long() # unnormalized coilID
    
#     latent_vol = embedding_vol[vol_ids]
#     latent_coil = embedding_coil[start_idx[vol_ids] + coil_ids]

#     # outputs = model(coords, latent_vol, latent_coil)
    
#     # predicted_volume[inputs_unnormalized[:,2], inputs_unnormalized[:,3], inputs_unnormalized[:,1], inputs_unnormalized[:,0]] = outputs
#     volume_kspace[inputs_unnormalized[:,2], inputs_unnormalized[:,3], inputs_unnormalized[:,1], inputs_unnormalized[:,0]] = targets

In [ ]:
n_slices, n_coils, height, width = shape

# Create tensors of indices for each dimension
kx_ids = torch.cat([torch.arange(left_idx), torch.arange(right_idx, width)])
# kx_ids = torch.arange(width)
ky_ids = torch.arange(height)
kz_ids = torch.arange(n_slices)
coil_ids = torch.arange(n_coils)

volume_kspace = torch.zeros(
    (n_slices, n_coils, height, width, 2),
    dtype=torch.float32,
)

kspace_ids = torch.meshgrid(kx_ids, ky_ids, kz_ids, coil_ids, indexing="ij")
kspace_ids = torch.stack(kspace_ids, dim=-1).reshape(-1, len(kspace_ids))

for points in dataloader:
    point_ids = points[1].int()
    targets = points[2]
    volume_kspace[
        point_ids[:, 2], point_ids[:, 3], point_ids[:, 1], point_ids[:, 0]
    ] = targets

In [ ]:
torch.view_as_complex(volume_kspace[0,0,:,left_idx:right_idx,:])

In [ ]:
vol_kspace[0,:,left_idx:right_idx]

In [ ]:
vol_img = rss(inverse_fft2_shift(torch.view_as_complex(volume_kspace)))
vol_kspace = fft2_shift(vol_img)
modulus = np.abs(rss(torch.view_as_complex(volume_kspace)))


plt.figure()
plt.subplot(1,2,1)
plt.imshow(np.abs(vol_img[0]), cmap='gray')
plt.subplot(1,2,2)
plt.imshow(modulus[0])

In [ ]:
predicted_volume = torch.load("./model_frozen_prediction_volID0.pth")

gt_volume_kspace = (
    volume_kspace * dataloader.dataset.metadata[vol_id]["norm_cste"]
)
predicted_kspace = (predicted_volume * dataloader.dataset.metadata[vol_id]["norm_cste"])
# predicted_kspace = predicted_volume.clone()

predicted_kspace = tensor_to_complex_np(predicted_kspace.detach().cpu())
gt_volume_kspace = tensor_to_complex_np(gt_volume_kspace)


predicted_kspace[..., left_idx:right_idx] = center_vals
gt_volume_kspace[..., left_idx:right_idx] = center_vals

predicted_img = rss(inverse_fft2_shift(predicted_kspace))
gt_img = rss(inverse_fft2_shift(gt_volume_kspace))

predicted_kspace = fft2_shift(predicted_img)
gt_kspace = fft2_shift(gt_img)
eps = 1.e-45

plt.figure(figsize=(15,10))
plt.subplot(1,2,1)
# plt.imshow(np.log(np.abs(predicted_kspace[0]) + eps))
plt.imshow(np.abs(predicted_img[1]), cmap='gray')
plt.axis('off')
plt.title('Prediction model')
plt.subplot(1,2,2)
plt.imshow(np.abs(gt_img[1]), cmap='gray')
plt.title('Ground truth')
# plt.subplot(1,2,2)
# plt.imshow(np.log(np.abs(gt_kspace[0]) + eps))
plt.axis('off')

In [ ]:
print(psnr(gt_img, predicted_img))
print(ssim(gt_img, predicted_img))

In [ ]:
gt_volume_kspace = (
    volume_kspace * dataloader.dataset.metadata[vol_id]["norm_cste"]
)
predicted_kspace = (predicted_volume * dataloader.dataset.metadata[vol_id]["norm_cste"])


predicted_kspace = tensor_to_complex_np(predicted_kspace.detach().cpu())
gt_volume_kspace = tensor_to_complex_np(gt_volume_kspace)


predicted_kspace[..., left_idx:right_idx] = center_vals
gt_volume_kspace[..., left_idx:right_idx] = center_vals

predicted_img = rss(inverse_fft2_shift(predicted_kspace))
gt_img = rss(inverse_fft2_shift(gt_volume_kspace))

predicted_kspace = fft2_shift(predicted_img)
gt_kspace = fft2_shift(gt_img)
eps = 1.e-45

plt.figure(figsize=(15,10))
plt.subplot(1,2,1)
plt.imshow(np.log(np.abs(predicted_kspace[0]) + eps))
# plt.imshow(np.abs(predicted_img[0]), cmap='gray')
plt.axis('off')
plt.title('Prediction model')
plt.subplot(1,2,2)
plt.colorbar()
# plt.imshow(np.abs(gt_img[0]), cmap='gray')
# plt.subplot(1,2,2)
plt.imshow(np.log(np.abs(gt_kspace[0]) + eps))
plt.title('Ground truth')
plt.axis('off')
plt.colorbar()

In [ ]:
torch.save(predicted_volume, "model_frozen_prediction_volID0.pth")

In [ ]:
dataset = KCoordDataset(path_to_data, n_volumes=n_volumes, n_slices=n_slices, with_mask=True, with_center=False, acceleration=4, center_frac=0.25)

dataloader_mask = DataLoader(
    dataset,
    batch_size=600_000,
    num_workers=0,
    shuffle=True,
)

vol_id = 0
shape = dataloader_mask.dataset.metadata[vol_id]["shape"]
center_data = dataloader_mask.dataset.metadata[vol_id]["center"]
left_idx, right_idx, center_vals = (
    center_data["left_idx"],
    center_data["right_idx"],
    center_data["vals"],
)
n_slices, n_coils, height, width = shape

volume_kspace_masked = torch.zeros(
    (n_slices, n_coils, height, width, 2),
    dtype=torch.float32,
)

for batch_idx, (inputs,inputs_unnormalized,targets) in enumerate(dataloader_mask):
    volume_kspace_masked[inputs_unnormalized[:,2], inputs_unnormalized[:,3], inputs_unnormalized[:,1], inputs_unnormalized[:,0]] = targets
    

In [ ]:
gt_volume_kspace = (
    volume_kspace_masked * dataloader_mask.dataset.metadata[vol_id]["norm_cste"]
)

gt_volume_kspace = tensor_to_complex_np(gt_volume_kspace)

# gt_volume_kspace[..., left_idx:right_idx] = center_vals


gt_img = rss(inverse_fft2_shift(gt_volume_kspace))

modulus_kspace = fft2_shift(gt_img)
# modulus_kspace[..., left_idx:right_idx] = 0

eps = 1.e-45

plt.figure(figsize=(15,10))
plt.subplot(1,2,1)
plt.imshow(np.log(np.abs(modulus_kspace[0])))
plt.axis('off')
plt.title('log(modulus)')
plt.colorbar()
plt.subplot(1,2,2)
plt.imshow(np.abs(gt_img[0]), cmap='gray')
# plt.subplot(1,2,2)
# plt.imshow(np.log(np.abs(gt_kspace[0]) + eps))
plt.axis('off')



In [ ]:
gt_modulus = np.abs(fft2_shift(gt_img))
predicted_modulus = np.abs(fft2_shift(predicted_img))


plt.figure(figsize=(15,10))
plt.subplot(1,2,1)
plt.imshow(np.log(gt_modulus[0] + eps))
plt.axis('off')
plt.title('Undersampled groundtruth')
plt.colorbar()
plt.subplot(1,2,2)
plt.imshow(np.log(predicted_modulus[0] + eps))
plt.axis('off')
plt.title('Undersampled predictions')
plt.colorbar()

In [ ]:
plt.figure(figsize=(10,10))

plt.subplot(2, 2, 1)
plt.imshow(np.log(gt_modulus[0] / dataloader.dataset.metadata[vol_id]["norm_cste"] + eps))
plt.axis('off')
plt.colorbar()
plt.title("Kspace")
plt.subplot(2, 2, 2)
plt.hist(np.log(gt_modulus[0].flatten()), log=True, bins=100)
plt.subplot(2, 2, 3)
plt.imshow(np.log(predicted_modulus[0] / dataloader.dataset.metadata[vol_id]["norm_cste"] + eps))
plt.axis('off')
plt.colorbar()
plt.title("Kspace")
plt.subplot(2, 2, 4)
plt.hist(np.log(predicted_modulus[0].flatten()), log=True, bins=100)
plt.show()


In [ ]:
vol_id = 0
file = dataloader.dataset.metadata[vol_id]["file"]
with h5py.File(file, "r") as hf:
    ground_truth = hf["reconstruction_rss"][()][: n_slices]
    
modulus = np.abs(fft2_shift(ground_truth))



In [ ]:
plt.imshow(np.log(modulus[0]) + eps)
# plt.imshow(ground_truth[0], cmap='gray')
plt.axis('off')
plt.colorbar()